***꼭 순서대로 실행해 주세요!!!!!!!***

# 기본, 재해싱

In [ ]:
class Node:
  def __init__(self, key, value):
    self.key = key
    self.value = value
    self.next = None

In [ ]:
class BaseHashTable:

  def __init__(self, size=13):
    self.size = size              # 테이블 크기
    self.count = 0                # 현재 저장된 데이터 개수
    self.table = [None] * size    # 해시 테이블 초기화


  def load_factor(self):
    return (self.count + 1) / self.size  # 적재율 (rehash 판단 기준) (rehash if문이 count하는 함수보다 )


  def rehash(self):
    old_table = self.table        # 기존 테이블 백업

    self.size *= 2                # 테이블 크기 2배 확장 (핵심)
    self.table = [None] * self.size
    self.count = 0                # 다시 삽입할 것이므로 count 초기화

    # 기존 데이터들을 새로운 테이블에 재삽입
    for item in old_table:

      if item is None:
        continue                  # 비어있는 슬롯은 무시

      # Chaining (list 형태) 처리
      if isinstance(item, list):
        for k, v in item:
          self.insert(k, v)       # 새 해시값으로 다시 삽입

      # Open Addressing (tuple 형태) 처리
      elif isinstance(item, tuple):
        self.insert(item[0], item[1])

      # Linked List (Chaining) 처리
      else:
        cur = item
        while cur:
          self.insert(cur.key, cur.value)  # 연결된 노드 모두 재삽입
          cur = cur.next

# 두 자릿수 접기

In [ ]:
class FoldingHashTable(BaseHashTable):

  def hash_fn(self, key):
    key = str(key)        # key를 문자열로 변환
    total = 0

    # 두 자리씩 끊어서 더하는 folding 방식
    for i in range(0, len(key), 2):
      total += int(key[i:i+2])

    return total % self.size   # 테이블 크기로 나눈 나머지


  def insert(self, key, value):

    if self.load_factor() > 0.7:   # 적재율 70% 초과 시
      self.rehash()                # 테이블 확장

    idx = self.hash_fn(key)        # 해시값 계산
    start = idx                    # 순환 탐사 시작점 저장

    # 충돌 발생 시 선형 탐사
    while self.table[idx] is not None:
      idx = (idx + 1) % self.size   # 끝 → 처음 순환

      if idx == start:              # 한 바퀴 돌면 실패
        print("insert FAIL: table is full")
        return

    self.table[idx] = (key, value)  # 삽입
    self.count += 1                 # 개수 증가


  def search(self, key):

    idx = self.hash_fn(key)
    start = idx

    while self.table[idx] is not None:

      if self.table[idx][0] == key:   # key 찾으면 반환
        return self.table[idx][1]

      idx = (idx + 1) % self.size     # 다음 슬롯 탐색

      if idx == start:                # 한 바퀴 돌면 종료
        break

    return None


  def delete(self, key):

    idx = self.hash_fn(key)
    start = idx

    while self.table[idx] is not None:

      if self.table[idx][0] == key:
        self.table[idx] = None        # 삭제
        self.count -= 1
        print(f"{key} deleted")
        return

      idx = (idx + 1) % self.size

      if idx == start:
        break

    print("delete FAIL: key not found")


  def display(self):
    print("\n[ 해시 테이블 상태 ]")
    for i, slot in enumerate(self.table):
      print(f"  [{i}] {slot}")
    print()

충돌 검증

In [ ]:
ht = FoldingHashTable(7)

keys = [1234, 3412, 5610, 1006]  # folding 결과 동일 → 충돌 유도

for k in keys:
  print(f"\ninsert({k})")
  ht.insert(k, chr(65 + keys.index(k)))
  ht.display()


insert(1234)

[ 해시 테이블 상태 ]
  [0] None
  [1] None
  [2] None
  [3] None
  [4] (1234, 'A')
  [5] None
  [6] None


insert(3412)

[ 해시 테이블 상태 ]
  [0] None
  [1] None
  [2] None
  [3] None
  [4] (1234, 'A')
  [5] (3412, 'B')
  [6] None


insert(5610)

[ 해시 테이블 상태 ]
  [0] None
  [1] None
  [2] None
  [3] (5610, 'C')
  [4] (1234, 'A')
  [5] (3412, 'B')
  [6] None


insert(1006)

[ 해시 테이블 상태 ]
  [0] None
  [1] None
  [2] (1006, 'D')
  [3] (5610, 'C')
  [4] (1234, 'A')
  [5] (3412, 'B')
  [6] None



재해싱 검증

In [ ]:
ht = FoldingHashTable(10)

keys = [1234, 3412, 2143, 4321, 1111, 2342, 4578, 9067]  # 5개 → rehash 발생

for i, k in enumerate(keys):
  print(f"\ninsert({k})")

  print("현재 load factor:", ht.load_factor())

  ht.insert(k, chr(65 + i))

  ht.display()


insert(1234)
현재 load factor: 0.1

[ 해시 테이블 상태 ]
  [0] None
  [1] None
  [2] None
  [3] None
  [4] None
  [5] None
  [6] (1234, 'A')
  [7] None
  [8] None
  [9] None


insert(3412)
현재 load factor: 0.2

[ 해시 테이블 상태 ]
  [0] None
  [1] None
  [2] None
  [3] None
  [4] None
  [5] None
  [6] (1234, 'A')
  [7] (3412, 'B')
  [8] None
  [9] None


insert(2143)
현재 load factor: 0.3

[ 해시 테이블 상태 ]
  [0] None
  [1] None
  [2] None
  [3] None
  [4] (2143, 'C')
  [5] None
  [6] (1234, 'A')
  [7] (3412, 'B')
  [8] None
  [9] None


insert(4321)
현재 load factor: 0.4

[ 해시 테이블 상태 ]
  [0] None
  [1] None
  [2] None
  [3] None
  [4] (2143, 'C')
  [5] (4321, 'D')
  [6] (1234, 'A')
  [7] (3412, 'B')
  [8] None
  [9] None


insert(1111)
현재 load factor: 0.5

[ 해시 테이블 상태 ]
  [0] None
  [1] None
  [2] (1111, 'E')
  [3] None
  [4] (2143, 'C')
  [5] (4321, 'D')
  [6] (1234, 'A')
  [7] (3412, 'B')
  [8] None
  [9] None


insert(2342)
현재 load factor: 0.6

[ 해시 테이블 상태 ]
  [0] None
  [1] None
  [2] (1111, 'E')
  [3] 

search & delete 검증

In [ ]:
# SEARCH
print("\n[SEARCH]")
for k in keys:
  print(f"search({k}) ->", ht.search(k))

# DELETE + DISPLAY
print("\n[DELETE]")
ht.delete(1234)
ht.display()

# SEARCH AFTER DELETE
print("\n[SEARCH AFTER DELETE]")
print("search(3412) ->", ht.search(3412))   # 삭제된 값
print("search(1234) ->", ht.search(1234))   # 정상 값


[SEARCH]
search(1234) -> A
search(3412) -> B
search(2143) -> C
search(4321) -> D
search(1111) -> E
search(2342) -> F
search(4578) -> G
search(9067) -> H

[DELETE]
1234 deleted

[ 해시 테이블 상태 ]
  [0] None
  [1] None
  [2] (1111, 'E')
  [3] (4578, 'G')
  [4] (2143, 'C')
  [5] (4321, 'D')
  [6] None
  [7] (3412, 'B')
  [8] (2342, 'F')
  [9] None
  [10] None
  [11] None
  [12] None
  [13] None
  [14] None
  [15] None
  [16] None
  [17] (9067, 'H')
  [18] None
  [19] None


[SEARCH AFTER DELETE]
search(3412) -> None
search(1234) -> None


# 여러 자리 자릿수 접기

In [ ]:
class NFoldingHashTable(BaseHashTable):

  def __init__(self, size, fold_size):
    super().__init__(size)
    self.fold_size = fold_size   # 접는 자릿수


  def hash_fn(self, key):
    key = str(key)
    total = 0

    # fold_size 만큼 끊기
    for i in range(0, len(key), self.fold_size):
      total += int(key[i:i+self.fold_size])

    return total % self.size


  def insert(self, key, value):

    if self.load_factor() > 0.7:
      self.rehash()

    idx = self.hash_fn(key)
    start = idx

    while self.table[idx] is not None:
      idx = (idx + 1) % self.size

      if idx == start:
        print("insert FAIL: table is full")
        return

    self.table[idx] = (key, value)
    self.count += 1


  def search(self, key):

    idx = self.hash_fn(key)
    start = idx

    while self.table[idx] is not None:

      if self.table[idx][0] == key:
        return self.table[idx][1]

      idx = (idx + 1) % self.size

      if idx == start:
        break

    return None


  def delete(self, key):

    idx = self.hash_fn(key)
    start = idx

    while self.table[idx] is not None:

      if self.table[idx][0] == key:
        self.table[idx] = None
        self.count -= 1
        print(f"{key} deleted")
        return

      idx = (idx + 1) % self.size

      if idx == start:
        break

    print("delete FAIL: key not found")


  def display(self):
    print("\n[ 해시 테이블 상태 ]")
    for i, slot in enumerate(self.table):
      print(f"  [{i}] {slot}")
    print()

충돌 검증

In [ ]:
ht = NFoldingHashTable(7, 3) #(hash table size, fold size)

keys = [1234, 3412, 5610, 1006]  # folding 결과 동일 → 충돌 유도

for k in keys:
  print(f"\ninsert({k})")
  ht.insert(k, chr(65 + keys.index(k)))
  ht.display()


insert(1234)

[ 해시 테이블 상태 ]
  [0] None
  [1] (1234, 'A')
  [2] None
  [3] None
  [4] None
  [5] None
  [6] None


insert(3412)

[ 해시 테이블 상태 ]
  [0] (3412, 'B')
  [1] (1234, 'A')
  [2] None
  [3] None
  [4] None
  [5] None
  [6] None


insert(5610)

[ 해시 테이블 상태 ]
  [0] (3412, 'B')
  [1] (1234, 'A')
  [2] (5610, 'C')
  [3] None
  [4] None
  [5] None
  [6] None


insert(1006)

[ 해시 테이블 상태 ]
  [0] (3412, 'B')
  [1] (1234, 'A')
  [2] (5610, 'C')
  [3] (1006, 'D')
  [4] None
  [5] None
  [6] None



재해싱 검증

In [ ]:
ht = NFoldingHashTable(7, 3)

keys = [1234, 3412, 2143, 4321, 1111]  # 5개 → rehash 발생

for i, k in enumerate(keys):
  print(f"\ninsert({k})")

  print("현재 load factor:", ht.load_factor())

  ht.insert(k, chr(65 + i))

  ht.display()


insert(1234)
현재 load factor: 0.14285714285714285

[ 해시 테이블 상태 ]
  [0] None
  [1] (1234, 'A')
  [2] None
  [3] None
  [4] None
  [5] None
  [6] None


insert(3412)
현재 load factor: 0.2857142857142857

[ 해시 테이블 상태 ]
  [0] (3412, 'B')
  [1] (1234, 'A')
  [2] None
  [3] None
  [4] None
  [5] None
  [6] None


insert(2143)
현재 load factor: 0.42857142857142855

[ 해시 테이블 상태 ]
  [0] (3412, 'B')
  [1] (1234, 'A')
  [2] (2143, 'C')
  [3] None
  [4] None
  [5] None
  [6] None


insert(4321)
현재 load factor: 0.5714285714285714

[ 해시 테이블 상태 ]
  [0] (3412, 'B')
  [1] (1234, 'A')
  [2] (2143, 'C')
  [3] None
  [4] None
  [5] None
  [6] (4321, 'D')


insert(1111)
현재 load factor: 0.7142857142857143

[ 해시 테이블 상태 ]
  [0] (1111, 'E')
  [1] (1234, 'A')
  [2] None
  [3] None
  [4] None
  [5] None
  [6] None
  [7] (3412, 'B')
  [8] (2143, 'C')
  [9] None
  [10] None
  [11] None
  [12] None
  [13] (4321, 'D')



search & delete 검증

In [ ]:
# SEARCH
print("\n[SEARCH]")
for k in keys:
  print(f"search({k}) ->", ht.search(k))

# DELETE + DISPLAY
print("\n[DELETE]")
ht.delete(1234)
ht.display()

# SEARCH AFTER DELETE
print("\n[SEARCH AFTER DELETE]")
print("search(3412) ->", ht.search(3412))   # 삭제된 값
print("search(1234) ->", ht.search(1234))   # 정상 값


[SEARCH]
search(1234) -> A
search(3412) -> B
search(2143) -> C
search(4321) -> D
search(1111) -> E

[DELETE]
1234 deleted

[ 해시 테이블 상태 ]
  [0] (1111, 'E')
  [1] None
  [2] None
  [3] None
  [4] None
  [5] None
  [6] None
  [7] (3412, 'B')
  [8] (2143, 'C')
  [9] None
  [10] None
  [11] None
  [12] None
  [13] (4321, 'D')


[SEARCH AFTER DELETE]
search(3412) -> B
search(1234) -> None


# 채이닝

In [ ]:
class ChainingHashTable(BaseHashTable):

  def hash_fn(self, key):
    return key % self.size


  def insert(self, key, value):

    if self.load_factor() > 0.7:
      self.rehash()

    idx = self.hash_fn(key)

    new = Node(key, value)

    if self.table[idx] is None:
      self.table[idx] = new
    else:
      cur = self.table[idx]
      new.next = cur # 기존에 while문을 통해 뒤에 붙이는 걸 앞에 붙이는 걸로 바꿈
      self.table[idx] = new

    self.count += 1


  def search(self, key):

    idx = self.hash_fn(key)
    cur = self.table[idx]

    while cur:
      if cur.key == key:
        return cur.value
      cur = cur.next

    return None


  def delete(self, key):

    idx = self.hash_fn(key)
    cur = self.table[idx]
    prev = None

    while cur:

      if cur.key == key:

        if prev is None:
          self.table[idx] = cur.next
        else:
          prev.next = cur.next

        self.count -= 1
        print(f"{key} deleted")
        return

      prev = cur
      cur = cur.next

    print("delete FAIL: key not found")


  def display(self):
    print("\n[ 해시 테이블 상태 ]")

    for i, slot in enumerate(self.table):

      print(f"[{i}]", end=" ")

      cur = slot
      if cur is None:
        print("None")
      else:
        while cur:
          print(f"({cur.key}, '{cur.value}')", end=" -> ")
          cur = cur.next
        print("None")

    print()

충돌 검증

In [ ]:
ht = ChainingHashTable(7)

keys = [10, 17, 24, 31]  # 모두 %7 = 3 → 완전 충돌

for i, k in enumerate(keys):
  print(f"\ninsert({k})")
  ht.insert(k, chr(65 + i))
  ht.display()


insert(10)

[ 해시 테이블 상태 ]
[0] None
[1] None
[2] None
[3] (10, 'A') -> None
[4] None
[5] None
[6] None


insert(17)

[ 해시 테이블 상태 ]
[0] None
[1] None
[2] None
[3] (17, 'B') -> (10, 'A') -> None
[4] None
[5] None
[6] None


insert(24)

[ 해시 테이블 상태 ]
[0] None
[1] None
[2] None
[3] (24, 'C') -> (17, 'B') -> (10, 'A') -> None
[4] None
[5] None
[6] None


insert(31)

[ 해시 테이블 상태 ]
[0] None
[1] None
[2] None
[3] (31, 'D') -> (24, 'C') -> (17, 'B') -> (10, 'A') -> None
[4] None
[5] None
[6] None



재해싱 검증

In [ ]:
ht = ChainingHashTable(7)

keys = [10, 17, 24, 31, 38]  # 5개 → rehash 발생

for i, k in enumerate(keys):

  print(f"\ninsert({k})")
  print("현재 load factor:", ht.load_factor())

  ht.insert(k, chr(65 + i))

  ht.display()


insert(10)
현재 load factor: 0.14285714285714285

[ 해시 테이블 상태 ]
[0] None
[1] None
[2] None
[3] (10, 'A') -> None
[4] None
[5] None
[6] None


insert(17)
현재 load factor: 0.2857142857142857

[ 해시 테이블 상태 ]
[0] None
[1] None
[2] None
[3] (17, 'B') -> (10, 'A') -> None
[4] None
[5] None
[6] None


insert(24)
현재 load factor: 0.42857142857142855

[ 해시 테이블 상태 ]
[0] None
[1] None
[2] None
[3] (24, 'C') -> (17, 'B') -> (10, 'A') -> None
[4] None
[5] None
[6] None


insert(31)
현재 load factor: 0.5714285714285714

[ 해시 테이블 상태 ]
[0] None
[1] None
[2] None
[3] (31, 'D') -> (24, 'C') -> (17, 'B') -> (10, 'A') -> None
[4] None
[5] None
[6] None


insert(38)
현재 load factor: 0.7142857142857143

[ 해시 테이블 상태 ]
[0] None
[1] None
[2] None
[3] (17, 'B') -> (31, 'D') -> None
[4] None
[5] None
[6] None
[7] None
[8] None
[9] None
[10] (38, 'E') -> (10, 'A') -> (24, 'C') -> None
[11] None
[12] None
[13] None



search & delete 검증

In [ ]:
# SEARCH
print("\n[SEARCH]")
for k in keys:
  print(f"search({k}) ->", ht.search(k))

# DELETE (중간 노드 삭제 테스트)
print("\n[DELETE]")
ht.delete(31)
ht.display()

# SEARCH AFTER DELETE
print("\n[SEARCH AFTER DELETE]")
print("search(17) ->", ht.search(17))   # 삭제된 값
print("search(10) ->", ht.search(10))   # 앞 노드
print("search(24) ->", ht.search(24))   # 뒤 노드


[SEARCH]
search(10) -> A
search(17) -> B
search(24) -> C
search(31) -> D
search(38) -> E

[DELETE]
31 deleted

[ 해시 테이블 상태 ]
[0] None
[1] None
[2] None
[3] (17, 'B') -> None
[4] None
[5] None
[6] None
[7] None
[8] None
[9] None
[10] (38, 'E') -> (10, 'A') -> (24, 'C') -> None
[11] None
[12] None
[13] None


[SEARCH AFTER DELETE]
search(17) -> B
search(10) -> A
search(24) -> C


# BST+체이닝(추가)

In [ ]:
class BSTNode:
  def __init__(self, key, value):
    self.key = key
    self.value = value
    self.left = None
    self.right = None
    self.next = None   #추가 (rehash용)

class BST:

  def insert(self, root, key, value):

    if root is None:
      return BSTNode(key, value)

    if key < root.key:
      root.left = self.insert(root.left, key, value)

    elif key > root.key:
      root.right = self.insert(root.right, key, value)

    else:
      root.value = value

    return root


  def search(self, root, key):

    if root is None:
      return None

    if key == root.key:
      return root.value

    if key < root.key:
      return self.search(root.left, key)

    return self.search(root.right, key)


  def find_min(self, node):
    while node.left is not None:
      node = node.left
    return node


  def delete(self, root, key):

    if root is None:
      return None

    if key < root.key:
      root.left = self.delete(root.left, key)

    elif key > root.key:
      root.right = self.delete(root.right, key)

    else:
      if root.left is None:
        return root.right

      if root.right is None:
        return root.left

      temp = self.find_min(root.right)
      root.key = temp.key
      root.value = temp.value
      root.right = self.delete(root.right, temp.key)

    return root


  def inorder(self, root):

    if root is None:
      return []

    return self.inorder(root.left) + [(root.key, root.value)] + self.inorder(root.right)


  # 핵심: next 연결 함수
  def link_nodes(self, root):

    if root is None:
      return

    nodes = []
    node_map = {}

    # 모든 노드 수집
    def collect(node):
      if node is None:
        return
      collect(node.left)
      nodes.append(node)
      collect(node.right)

    collect(root)

    # next 연결
    for i in range(len(nodes)-1):
      nodes[i].next = nodes[i+1]

    if nodes:
      nodes[-1].next = None

class BSTChainingHashTable(BaseHashTable):

  def __init__(self, size=7):
    super().__init__(size)
    self.bst = BST()


  def hash_fn(self, key):
    return key % self.size


  def insert(self, key, value):

    if self.load_factor() > 0.7:
      print("\n*** REHASH 발생 ***")

      # rehash 전에 next 연결
      for root in self.table:
        if root is not None:
          self.bst.link_nodes(root)

      self.rehash()

    idx = self.hash_fn(key)

    self.table[idx] = self.bst.insert(self.table[idx], key, value)

    # 삽입 후 next 다시 연결
    self.bst.link_nodes(self.table[idx])

    self.count += 1


  def search(self, key):

    idx = self.hash_fn(key)
    return self.bst.search(self.table[idx], key)


  def delete(self, key):

    idx = self.hash_fn(key)

    if self.search(key) is None:
      print("delete FAIL: key not found")
      return

    self.table[idx] = self.bst.delete(self.table[idx], key)

    # 삭제 후 next 재연결
    if self.table[idx] is not None:
      self.bst.link_nodes(self.table[idx])

    self.count -= 1
    print(f"{key} deleted")


  def display(self):

    print("\n[ 해시 테이블 상태 ]")

    for i, root in enumerate(self.table):

      if root is None:
        print(f"[{i}] None")

      else:
        nodes = self.bst.inorder(root)
        print(f"[{i}] {nodes}")

    print()

충돌 검증

In [ ]:
ht = BSTChainingHashTable(7)

keys = [10, 24, 17]  # 모두 3번 슬롯 (충돌)

for i, k in enumerate(keys):
  print(f"\ninsert({k})")
  ht.insert(k, chr(65+i))
  ht.display()


insert(10)

[ 해시 테이블 상태 ]
[0] None
[1] None
[2] None
[3] [(10, 'A')]
[4] None
[5] None
[6] None


insert(24)

[ 해시 테이블 상태 ]
[0] None
[1] None
[2] None
[3] [(10, 'A'), (24, 'B')]
[4] None
[5] None
[6] None


insert(17)

[ 해시 테이블 상태 ]
[0] None
[1] None
[2] None
[3] [(10, 'A'), (17, 'C'), (24, 'B')]
[4] None
[5] None
[6] None



재해싱 검증

In [ ]:
ht = BSTChainingHashTable(7)

keys = [10, 17, 24, 31, 38, 45]  # 6개 → rehash 발생

for i, k in enumerate(keys):
  print(f"\ninsert({k})")
  print("load factor:", ht.load_factor())
  ht.insert(k, chr(65+i))
  ht.display()


insert(10)
load factor: 0.14285714285714285

[ 해시 테이블 상태 ]
[0] None
[1] None
[2] None
[3] [(10, 'A')]
[4] None
[5] None
[6] None


insert(17)
load factor: 0.2857142857142857

[ 해시 테이블 상태 ]
[0] None
[1] None
[2] None
[3] [(10, 'A'), (17, 'B')]
[4] None
[5] None
[6] None


insert(24)
load factor: 0.42857142857142855

[ 해시 테이블 상태 ]
[0] None
[1] None
[2] None
[3] [(10, 'A'), (17, 'B'), (24, 'C')]
[4] None
[5] None
[6] None


insert(31)
load factor: 0.5714285714285714

[ 해시 테이블 상태 ]
[0] None
[1] None
[2] None
[3] [(10, 'A'), (17, 'B'), (24, 'C'), (31, 'D')]
[4] None
[5] None
[6] None


insert(38)
load factor: 0.7142857142857143

*** REHASH 발생 ***

[ 해시 테이블 상태 ]
[0] None
[1] None
[2] None
[3] [(17, 'B'), (31, 'D')]
[4] None
[5] None
[6] None
[7] None
[8] None
[9] None
[10] [(10, 'A'), (24, 'C'), (38, 'E')]
[11] None
[12] None
[13] None


insert(45)
load factor: 0.42857142857142855

[ 해시 테이블 상태 ]
[0] None
[1] None
[2] None
[3] [(17, 'B'), (31, 'D'), (45, 'F')]
[4] None
[5] None
[6] None
[7] 

search & delete 검증

In [ ]:
print("\n[SEARCH]")

for k in [10, 17, 24, 31, 38, 45]:
  print(f"search({k}) ->", ht.search(k))


print("\n[DELETE]")

ht.delete(24)
ht.display()

ht.delete(10)
ht.display()


print("\n[SEARCH AFTER DELETE]")

print("search(24) ->", ht.search(24))  # None
print("search(10) ->", ht.search(10))  # None
print("search(17) ->", ht.search(17))  # 남아있음


[SEARCH]
search(10) -> A
search(17) -> B
search(24) -> C
search(31) -> D
search(38) -> E
search(45) -> F

[DELETE]
24 deleted

[ 해시 테이블 상태 ]
[0] None
[1] None
[2] None
[3] [(17, 'B'), (31, 'D'), (45, 'F')]
[4] None
[5] None
[6] None
[7] None
[8] None
[9] None
[10] [(10, 'A'), (38, 'E')]
[11] None
[12] None
[13] None

10 deleted

[ 해시 테이블 상태 ]
[0] None
[1] None
[2] None
[3] [(17, 'B'), (31, 'D'), (45, 'F')]
[4] None
[5] None
[6] None
[7] None
[8] None
[9] None
[10] [(38, 'E')]
[11] None
[12] None
[13] None


[SEARCH AFTER DELETE]
search(24) -> None
search(10) -> None
search(17) -> B


# 선형 탐사

In [ ]:
class LinearProbingHashTable(BaseHashTable):

  def hash_fn(self, key):
    return key % self.size   # key를 테이블 크기로 나눈 나머지


  def insert(self, key, value):

    # load factor가 0.7 초과하면 rehash
    if self.load_factor() > 0.7:
      print("\n*** REHASH 발생 ***")
      self.rehash()

    idx = self.hash_fn(key)  # 시작 index
    start = idx              # 한 바퀴 체크용

    # 충돌 발생 시 선형 탐사 (한 칸씩 이동)
    while self.table[idx] is not None:
      idx = (idx + 1) % self.size   # 다음 칸

      if idx == start:              # 한 바퀴 돌면 실패
        print("insert FAIL: table is full")
        return

    self.table[idx] = (key, value)  # 삽입
    self.count += 1


  def search(self, key):

    idx = self.hash_fn(key)
    start = idx

    # 값이 존재하는 동안 계속 탐색
    while self.table[idx] is not None:

      if self.table[idx][0] == key:   # key 찾으면 반환
        return self.table[idx][1]

      idx = (idx + 1) % self.size

      if idx == start:
        break

    return None   # 못 찾으면 None


  def delete(self, key):

    idx = self.hash_fn(key)
    start = idx

    while self.table[idx] is not None:

      if self.table[idx][0] == key:
        self.table[idx] = None
        self.count -= 1
        print(f"{key} deleted")
        return

      idx = (idx + 1) % self.size

      if idx == start:
        break

    print("delete FAIL: key not found")


  def display(self):
    print("\n[ 해시 테이블 상태 ]")
    for i, slot in enumerate(self.table):
      print(f"  [{i}] {slot}")
    print()

충돌 검증

In [ ]:
ht = LinearProbingHashTable(7)

keys = [10, 17, 24]  # 모두 index 3 → 충돌 발생

for i, k in enumerate(keys):
  print(f"\ninsert({k})")
  ht.insert(k, chr(65+i))
  ht.display()


insert(10)

[ 해시 테이블 상태 ]
  [0] None
  [1] None
  [2] None
  [3] (10, 'A')
  [4] None
  [5] None
  [6] None


insert(17)

[ 해시 테이블 상태 ]
  [0] None
  [1] None
  [2] None
  [3] (10, 'A')
  [4] (17, 'B')
  [5] None
  [6] None


insert(24)

[ 해시 테이블 상태 ]
  [0] None
  [1] None
  [2] None
  [3] (10, 'A')
  [4] (17, 'B')
  [5] (24, 'C')
  [6] None



재해싱 검증

In [ ]:
ht = LinearProbingHashTable(7)

keys = [10, 17, 24, 31, 38, 45]  # 6개 → rehash

for i, k in enumerate(keys):
  print(f"\ninsert({k})")
  print("load factor:", ht.load_factor())
  ht.insert(k, chr(65+i))
  ht.display()


insert(10)
load factor: 0.14285714285714285

[ 해시 테이블 상태 ]
  [0] None
  [1] None
  [2] None
  [3] (10, 'A')
  [4] None
  [5] None
  [6] None


insert(17)
load factor: 0.2857142857142857

[ 해시 테이블 상태 ]
  [0] None
  [1] None
  [2] None
  [3] (10, 'A')
  [4] (17, 'B')
  [5] None
  [6] None


insert(24)
load factor: 0.42857142857142855

[ 해시 테이블 상태 ]
  [0] None
  [1] None
  [2] None
  [3] (10, 'A')
  [4] (17, 'B')
  [5] (24, 'C')
  [6] None


insert(31)
load factor: 0.5714285714285714

[ 해시 테이블 상태 ]
  [0] None
  [1] None
  [2] None
  [3] (10, 'A')
  [4] (17, 'B')
  [5] (24, 'C')
  [6] (31, 'D')


insert(38)
load factor: 0.7142857142857143

*** REHASH 발생 ***

[ 해시 테이블 상태 ]
  [0] None
  [1] None
  [2] None
  [3] (17, 'B')
  [4] (31, 'D')
  [5] None
  [6] None
  [7] None
  [8] None
  [9] None
  [10] (10, 'A')
  [11] (24, 'C')
  [12] (38, 'E')
  [13] None


insert(45)
load factor: 0.42857142857142855

[ 해시 테이블 상태 ]
  [0] None
  [1] None
  [2] None
  [3] (17, 'B')
  [4] (31, 'D')
  [5] (45, 'F

search & delete 검증

In [ ]:
print("\n[SEARCH]")

for k in [10, 17, 24]:
  print(f"search({k}) ->", ht.search(k))


print("\n[DELETE]")

ht.delete(10)
ht.display()


print("\n[SEARCH AFTER DELETE]")

print("search(17) ->", ht.search(17))
print("search(24) ->", ht.search(24))


[SEARCH]
search(10) -> A
search(17) -> B
search(24) -> C

[DELETE]
10 deleted

[ 해시 테이블 상태 ]
  [0] None
  [1] None
  [2] None
  [3] (17, 'B')
  [4] (31, 'D')
  [5] (45, 'F')
  [6] None
  [7] None
  [8] None
  [9] None
  [10] None
  [11] (24, 'C')
  [12] (38, 'E')
  [13] None


[SEARCH AFTER DELETE]
search(17) -> B
search(24) -> None


# 제곱 탐사

In [ ]:
class QuadraticProbingHashTable(BaseHashTable):

  def hash_fn(self, key):
    return key % self.size


  def insert(self, key, value):

    if self.load_factor() > 0.7:
      self.rehash()

    idx = self.hash_fn(key)
    i = 1
    start = idx

    while self.table[idx] is not None:
      idx = (idx + i*i) % self.size
      i += 1

    self.table[idx] = (key, value)
    self.count += 1


  def search(self, key):

    idx = self.hash_fn(key)
    i = 1
    start = idx

    while self.table[idx] is not None:

      if self.table[idx][0] == key:
        return self.table[idx][1]

      idx = (self.hash_fn(key) + i*i) % self.size
      i += 1

      if idx == start:
        break

    return None


  def delete(self, key):

    idx = self.hash_fn(key)
    i = 1
    start = idx

    while self.table[idx] is not None:

      if self.table[idx][0] == key:
        self.table[idx] = None
        self.count -= 1
        print(f"{key} deleted")
        return

      idx = (self.hash_fn(key) + i*i) % self.size
      i += 1

      if idx == start:
        break

    print("delete FAIL: key not found")


  def display(self):
    print("\n[ 해시 테이블 상태 ]")
    for i, slot in enumerate(self.table):
      print(f"  [{i}] {slot}")
    print()

충돌 검증

In [ ]:
ht = QuadraticProbingHashTable(7)

keys = [10, 17, 24]  # 모두 hash = 3

for i, k in enumerate(keys):
  print(f"\ninsert({k})")
  ht.insert(k, chr(65+i))
  ht.display()


insert(10)

[ 해시 테이블 상태 ]
  [0] None
  [1] None
  [2] None
  [3] (10, 'A')
  [4] None
  [5] None
  [6] None


insert(17)

[ 해시 테이블 상태 ]
  [0] None
  [1] None
  [2] None
  [3] (10, 'A')
  [4] (17, 'B')
  [5] None
  [6] None


insert(24)

[ 해시 테이블 상태 ]
  [0] None
  [1] (24, 'C')
  [2] None
  [3] (10, 'A')
  [4] (17, 'B')
  [5] None
  [6] None



재해싱 검증

In [ ]:
ht = QuadraticProbingHashTable(7)

keys = [10, 17, 76, 31, 66, 45]

for i, k in enumerate(keys):
  print(f"\ninsert({k})")
  print("load factor:", ht.load_factor())
  ht.insert(k, chr(65+i))
  ht.display()


insert(10)
load factor: 0.14285714285714285

[ 해시 테이블 상태 ]
  [0] None
  [1] None
  [2] None
  [3] (10, 'A')
  [4] None
  [5] None
  [6] None


insert(17)
load factor: 0.2857142857142857

[ 해시 테이블 상태 ]
  [0] None
  [1] None
  [2] None
  [3] (10, 'A')
  [4] (17, 'B')
  [5] None
  [6] None


insert(76)
load factor: 0.42857142857142855

[ 해시 테이블 상태 ]
  [0] None
  [1] None
  [2] None
  [3] (10, 'A')
  [4] (17, 'B')
  [5] None
  [6] (76, 'C')


insert(31)
load factor: 0.5714285714285714

[ 해시 테이블 상태 ]
  [0] None
  [1] (31, 'D')
  [2] None
  [3] (10, 'A')
  [4] (17, 'B')
  [5] None
  [6] (76, 'C')


insert(66)
load factor: 0.7142857142857143

[ 해시 테이블 상태 ]
  [0] None
  [1] None
  [2] None
  [3] (31, 'D')
  [4] (17, 'B')
  [5] None
  [6] (76, 'C')
  [7] None
  [8] None
  [9] None
  [10] (10, 'A')
  [11] (66, 'E')
  [12] None
  [13] None


insert(45)
load factor: 0.42857142857142855

[ 해시 테이블 상태 ]
  [0] None
  [1] None
  [2] None
  [3] (31, 'D')
  [4] (17, 'B')
  [5] None
  [6] (76, 'C')
  [7]

search & delete 검증

In [ ]:
print("\n[SEARCH]")

for k in [10, 17, 24]:
  print(f"search({k}) ->", ht.search(k))


print("\n[DELETE]")

ht.delete(10)   # 첫 위치 삭제
ht.display()


print("\n[SEARCH AFTER DELETE]")

print("search(17) ->", ht.search(17))
print("search(24) ->", ht.search(24))


[SEARCH]
search(10) -> A
search(17) -> B
search(24) -> None

[DELETE]
10 deleted

[ 해시 테이블 상태 ]
  [0] None
  [1] None
  [2] None
  [3] (31, 'D')
  [4] (17, 'B')
  [5] None
  [6] (76, 'C')
  [7] None
  [8] (45, 'F')
  [9] None
  [10] None
  [11] (66, 'E')
  [12] None
  [13] None


[SEARCH AFTER DELETE]
search(17) -> B
search(24) -> None


# 이중 해싱

In [ ]:
class DoubleHashingHashTable(BaseHashTable):

  def h1(self, key):
    return key % self.size   # 기본 해시 함수

  def h2(self, key):
    return key % 3 +2    # 두 번째 해시 (step size)


  def insert(self, key, value):

    # load factor가 0.7 초과하면 재해싱
    if self.load_factor() > 0.7:
      print("\n*** REHASH 발생 ***")
      self.rehash()

    idx = self.h1(key)       # 시작 위치
    step = self.h2(key)      # 이동 간격
    start = idx              # 한 바퀴 체크용

    # 충돌 발생 시 step만큼 점프
    while self.table[idx] is not None:
      idx = (idx + step) % self.size

      if idx == start:
        print("insert FAIL: table is full")
        return

    self.table[idx] = (key, value)
    self.count += 1


  def search(self, key):

    idx = self.h1(key)
    step = self.h2(key)
    start = idx

    while self.table[idx] is not None:

      if self.table[idx][0] == key:
        return self.table[idx][1]

      idx = (idx + step) % self.size

      if idx == start:
        break

    return None


  def delete(self, key):

    idx = self.h1(key)
    step = self.h2(key)
    start = idx

    while self.table[idx] is not None:

      if self.table[idx][0] == key:
        self.table[idx] = None
        self.count -= 1
        print(f"{key} deleted")
        return

      idx = (idx + step) % self.size

      if idx == start:
        break

    print("delete FAIL: key not found")


  def display(self):
    print("\n[ 해시 테이블 상태 ]")
    for i, slot in enumerate(self.table):
      print(f"  [{i}] {slot}")
    print()

충돌 검증

In [ ]:
ht = DoubleHashingHashTable(7)

keys = [9, 16, 23, 37]  # 모두 h1 = 2

for i, k in enumerate(keys):
  print(f"\ninsert({k})")
  ht.insert(k, chr(65+i))
  ht.display()


insert(9)

[ 해시 테이블 상태 ]
  [0] None
  [1] None
  [2] (9, 'A')
  [3] None
  [4] None
  [5] None
  [6] None


insert(16)

[ 해시 테이블 상태 ]
  [0] None
  [1] None
  [2] (9, 'A')
  [3] None
  [4] None
  [5] (16, 'B')
  [6] None


insert(23)

[ 해시 테이블 상태 ]
  [0] None
  [1] None
  [2] (9, 'A')
  [3] None
  [4] None
  [5] (16, 'B')
  [6] (23, 'C')


insert(37)

[ 해시 테이블 상태 ]
  [0] None
  [1] (37, 'D')
  [2] (9, 'A')
  [3] None
  [4] None
  [5] (16, 'B')
  [6] (23, 'C')



재해싱 검증

In [ ]:
ht = DoubleHashingHashTable(7)

keys = [10, 17, 24, 31, 38, 45]

for i, k in enumerate(keys):
  print(f"\ninsert({k})")
  print("load factor:", ht.load_factor())
  ht.insert(k, chr(65+i))
  ht.display()


insert(10)
load factor: 0.14285714285714285

[ 해시 테이블 상태 ]
  [0] None
  [1] None
  [2] None
  [3] (10, 'A')
  [4] None
  [5] None
  [6] None


insert(17)
load factor: 0.2857142857142857

[ 해시 테이블 상태 ]
  [0] (17, 'B')
  [1] None
  [2] None
  [3] (10, 'A')
  [4] None
  [5] None
  [6] None


insert(24)
load factor: 0.42857142857142855

[ 해시 테이블 상태 ]
  [0] (17, 'B')
  [1] None
  [2] None
  [3] (10, 'A')
  [4] None
  [5] (24, 'C')
  [6] None


insert(31)
load factor: 0.5714285714285714

[ 해시 테이블 상태 ]
  [0] (17, 'B')
  [1] None
  [2] None
  [3] (10, 'A')
  [4] None
  [5] (24, 'C')
  [6] (31, 'D')


insert(38)
load factor: 0.7142857142857143

*** REHASH 발생 ***

[ 해시 테이블 상태 ]
  [0] (38, 'E')
  [1] None
  [2] None
  [3] (17, 'B')
  [4] None
  [5] None
  [6] (31, 'D')
  [7] None
  [8] None
  [9] None
  [10] (10, 'A')
  [11] None
  [12] (24, 'C')
  [13] None


insert(45)
load factor: 0.42857142857142855

[ 해시 테이블 상태 ]
  [0] (38, 'E')
  [1] None
  [2] None
  [3] (17, 'B')
  [4] None
  [5] (45, 'F

search & delete 검증

In [ ]:
print("\n[SEARCH]")

for k in [10, 17, 24]:
  print(f"search({k}) ->", ht.search(k))


print("\n[DELETE]")

ht.delete(10)
ht.display()


print("\n[SEARCH AFTER DELETE]")

print("search(17) ->", ht.search(17))
print("search(24) ->", ht.search(24))


[SEARCH]
search(10) -> A
search(17) -> B
search(24) -> C

[DELETE]
10 deleted

[ 해시 테이블 상태 ]
  [0] (38, 'E')
  [1] None
  [2] None
  [3] (17, 'B')
  [4] None
  [5] (45, 'F')
  [6] (31, 'D')
  [7] None
  [8] None
  [9] None
  [10] None
  [11] None
  [12] (24, 'C')
  [13] None


[SEARCH AFTER DELETE]
search(17) -> B
search(24) -> None
